In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import TensorDataset, DataLoader

# 공통 변수
selected_cols = [
    '누적량1',
    '현재EC(dS)',
    '현재PH(pH)',
    '현재일사(W)',
    '누적일사(J)',
    'J/Day'
]
seq_length = 30

# 1. 시퀀스 생성 함수
def create_sequences(data, seq_len=30):
    sequences = []
    targets = []
    for i in range(len(data) - seq_len):
        sequences.append(data[i:i+seq_len])
        targets.append(data[i+seq_len])
    return np.array(sequences), np.array(targets)

# 2. 전처리 함수
def preprocess_csv(csv_path, selected_cols, scaler=None, fit_scaler=False):
    df = pd.read_csv(csv_path, low_memory=False)
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    df = df.sort_values('date').reset_index(drop=True)

    df_numeric = df[selected_cols].copy()
    df_numeric = df_numeric.apply(pd.to_numeric, errors='coerce')
    df_numeric = df_numeric.infer_objects(copy=False)
    df_numeric.interpolate(method='linear', limit_direction='both', inplace=True)
    df_numeric = df_numeric.clip(lower=df_numeric.quantile(0.01), upper=df_numeric.quantile(0.99), axis=1)

    if fit_scaler:
        scaler = StandardScaler()
        scaled = scaler.fit_transform(df_numeric)
    else:
        scaled = scaler.transform(df_numeric)

    return scaled, scaler

# 3. 1번 농가 데이터 로드 및 시퀀스 생성
data1_scaled, scaler = preprocess_csv(
    '/content/drive/MyDrive/Colab Notebooks/content/TS_Timeseries/tom1/토마토_환경데이터_1번농가(그린CS).csv',
    selected_cols,
    fit_scaler=True
)
X_1, y_1 = create_sequences(data1_scaled, seq_length)

# 4. 4번 농가 데이터 로드 및 시퀀스 생성 (동일 스케일러로 정규화)
data4_scaled, _ = preprocess_csv(
    '/content/drive/MyDrive/Colab Notebooks/content/TS_Timeseries/tom4/토마토_환경데이터_4번농가(그린CS).csv',
    selected_cols,
    scaler=scaler,
    fit_scaler=False
)
X_4, y_4 = create_sequences(data4_scaled, seq_length)

# 5. 병합
X_all = np.concatenate([X_1, X_4], axis=0)
y_all = np.concatenate([y_1, y_4], axis=0)

# 6. DataLoader 생성
X_tensor = torch.tensor(X_all, dtype=torch.float32)
y_tensor = torch.tensor(y_all, dtype=torch.float32)
dataset_all = TensorDataset(X_tensor, y_tensor)
dataloader_all = DataLoader(dataset_all, batch_size=64, shuffle=True)

print("DataLoader 준비 완료 ✅")
print(f"총 시퀀스 수: {len(dataset_all)} | 입력 shape: {X_tensor.shape} | 출력 shape: {y_tensor.shape}")


DataLoader 준비 완료 ✅
총 시퀀스 수: 354267 | 입력 shape: torch.Size([354267, 30, 6]) | 출력 shape: torch.Size([354267, 6])


# **1. 기존 모델 로드**

먼저 1번 농가 데이터로 이미 학습되어 저장된 모델을 불러와서, 이 모델을 기반으로 추가 학습을 진행해야 해. 모델 아키텍처는 그대로 유지하면서 state_dict를 로드한다.

In [13]:
import torch
import torch.nn as nn
import math

# Positional Encoding 클래스 (이미 제공된 것 활용)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)  # [max_len, 1, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: [seq_len, batch_size, d_model]
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

# Transformer 기반 시계열 모델 (다중 센서 예측 버전)
class TransformerTimeSeriesMulti(nn.Module):
    def __init__(self, input_dim, feature_size=64, num_layers=2, nhead=4, dropout=0.1, output_dim=None):
        super(TransformerTimeSeriesMulti, self).__init__()
        if output_dim is None:
            output_dim = input_dim  # 보통 입력 센서 수와 동일하게 출력

        self.input_linear = nn.Linear(input_dim, feature_size)
        self.pos_encoder = PositionalEncoding(feature_size, dropout)
        encoder_layers = nn.TransformerEncoderLayer(d_model=feature_size, nhead=nhead, dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layers, num_layers)
        self.decoder = nn.Linear(feature_size, output_dim)
        self.feature_size = feature_size

    def forward(self, src):
        # src: [batch_size, seq_len, input_dim]
        x = self.input_linear(src)            # → [batch_size, seq_len, feature_size]
        x = x.transpose(0, 1)                 # → [seq_len, batch_size, feature_size]
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)       # → [seq_len, batch_size, feature_size]
        # 마지막 타임스텝의 출력 사용
        out = self.decoder(x[-1, :, :])        # → [batch_size, output_dim]
        return out


In [14]:
import torch

# 예시: 이미 학습된 1번 농가 모델의 checkpoint 경로
model = TransformerTimeSeriesMulti(input_dim=6, feature_size=64, num_layers=2, nhead=4, dropout=0.1, output_dim=6)
model.load_state_dict(torch.load("/content/drive/MyDrive/Colab Notebooks/Tomato_Timeseries/greenscs_trained.pt"))
model.train()  # 계속 학습 모드로 설정

/usr/local/lib/python3.11/dist-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


TransformerTimeSeriesMulti(
  (input_linear): Linear(in_features=6, out_features=64, bias=True)
  (pos_encoder): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=2048, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=2048, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (decoder): Linear(in_features=64, out_features=6, bias=True)
)

In [1]:
# 이어서 학습 진행
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = torch.nn.MSELoss()
epochs = 10

for epoch in range(epochs):
    total_loss = 0.0
    for xb, yb in dataloader_all:
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
    avg_loss = total_loss / len(dataset_all)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.6f}")


NameError: name 'torch' is not defined

# **5. 모델 추가 학습 (Fine-tuning)**

이제 불러온 기존 모델을 4번 농가 데이터로 추가 학습하는데, 두 가지 접근이 있다.

# 5.1 선택 A: 오직 4번 농가 데이터만 이용하여 Fine-tuning
1번 모델을 그대로 사용하되, 현재 4번 데이터만 fine-tuning 해서 모델이 새로운 환경에 적응하도록 한다.

In [ ]:
import torch.optim as optim
import torch.nn as nn

# 옵티마이저와 손실 함수 설정 (원하는 학습률로 조정)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.MSELoss()

# Fine-tuning 루프: 4번 농가 데이터로만 학습
model.train()
epochs = 10  # 실험에 따라 조정

for epoch in range(epochs):
    epoch_loss = 0.0
    for xb, yb in dataloader4:
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    avg_loss = epoch_loss / len(dataset4)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.6f}")

# 5.2 선택 B: 1번과 4번 데이터를 모두 혼합하여 학습
만약 1번 데이터의 특성을 유지하면서 4번 데이터를 포함시키고 싶다면,

두 데이터셋을 병합하여 학습을 진행하는 방법도 있다.

선택 A vs. 선택 B

선택 A는 "도메인 적응"에 집중할 수 있고, 1번 데이터의 영향은 그대로 유지한 상태에서 새로운 환경에 빠르게 적응할 때 효과적.

선택 B는 모델이 두 농가의 공통 패턴을 학습하는 데 도움이 되어 더 범용적인 모델이 될 수 있음.

In [11]:
# 예를 들어, X_all과 y_all을 만들어 병합함
# X_1, y_1는 기존 1번 농가 데이터 시퀀스였다고 가정
X_all = np.concatenate([X_1, X4], axis=0)
y_all = np.concatenate([y_1, y4], axis=0)

X_all_tensor = torch.tensor(X_all, dtype=torch.float32)
y_all_tensor = torch.tensor(y_all, dtype=torch.float32)

dataset_all = TensorDataset(X_all_tensor, y_all_tensor)
dataloader_all = DataLoader(dataset_all, batch_size=64, shuffle=True)

# 이어서 모델 추가 학습
model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    for xb, yb in dataloader_all:
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    avg_loss = epoch_loss / len(dataset_all)
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.6f}")

NameError: name 'X_1' is not defined

# **6. 모델 저장 및 향후 평가**
4번 농가 또는 전체 데이터로 추가 학습한 후, 성능을 검증하고 모델을 저장하면 된다.

In [ ]:
torch.save(model.state_dict(), "/content/drive/MyDrive/Colab Notebooks/Tomato_Timeseries/greenscs_finetuned.pt")

# **요약**
기존 모델 불러오기: 1번 농가 학습된 모델 로드

4번 농가 데이터 전처리: 동일한 센서, 결측치 보간, 정규화, 시퀀스 생성

DataLoader 구성: 4번 농가 데이터를 PyTorch DataLoader로 준비

모델 추가 학습:

선택 A: 4번 농가 데이터만으로 fine-tuning

선택 B: 1번과 4번 데이터 병합하여 학습

평가 및 저장: 학습 후 테스트, 모델 저장 및 향후 Xiaomi 데이터 전이학습 준비

이 단계들을 차례대로 진행하면, 기존 모델에 새로운 환경 데이터(4번 농가)를 효과적으로 통합할 수 있고, 이후 Xiaomi 데이터 전이학습으로 확장하기 위한 탄탄한 기반이 될 거야.